# 02.4 — PyTorch tensors intro

`torch.Tensor` is NumPy's `ndarray` with three additions: **device placement** (CPU/GPU), **autograd** (gradient tracking — covered in [02.5](02.5_autograd_basics.ipynb)), and a neural-net-focused operator set. If you're comfortable with the NumPy material in [02.1](02.1_numpy_vs_matlab_arrays.ipynb), 90% transfers directly.

This notebook covers:

* Creating tensors; converting to/from NumPy (and when memory is shared).
* dtypes — why `float32` is the deep-learning default (not MATLAB's `double`).
* Devices — CPU, CUDA, Apple MPS; moving tensors between them.
* The in-place-operation convention (`add_` vs `add`).

**Prerequisite:** [02.1](02.1_numpy_vs_matlab_arrays.ipynb).

## Section 1 — What MATLAB does

MATLAB's deep-learning arrays are `dlarray` (with optional labels like `'SSCTB'`) wrapped over `gpuArray` when on GPU:

```matlab
X = randn(58, 100, 6, 'single');   % note: 'single' = float32
dlX = dlarray(X, 'SSCB');           % labeled deep-learning array
dlX = gpuArray(dlX);                % move to GPU
```

Three MATLAB habits to notice:

1. MATLAB defaults to `double` (float64) but deep learning code explicitly asks for `'single'` — the toolbox trains in float32.
2. `gpuArray` / `gather` move data on/off the GPU explicitly.
3. `dlarray`'s dimension labels do the job that axis-order conventions do in PyTorch ([02.2](02.2_array_axis_conventions.ipynb)).

PyTorch's model is the same shape, different names: `dtype=torch.float32`, `.to(device)`, `.cpu()`.

## Section 2 — The Python concepts you need

### 2.1 — Creating tensors

In [ ]:
import torch
import numpy as np

# From Python lists
t = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(t)
print(t.shape, t.dtype)

In [ ]:
# The same constructors you know from NumPy
print(torch.zeros(2, 3))
print(torch.ones(2, 3))
print(torch.arange(0, 10, 2))
print(torch.linspace(0, 1, 5))
g = torch.Generator().manual_seed(0)
print(torch.randn(2, 3, generator=g))    # standard normal, seeded

**Small API differences from NumPy:**

| NumPy | PyTorch | Note |
|---|---|---|
| `np.zeros((2, 3))` | `torch.zeros(2, 3)` | torch takes bare ints, no tuple needed (tuple also works) |
| `np.random.standard_normal(...)` | `torch.randn(...)` | shorter name |
| `arr.shape` | `t.shape` or `t.size()` | both work |
| `arr.reshape(...)` | `t.reshape(...)` or `t.view(...)` | `view` requires contiguous memory |
| `np.concatenate` | `torch.cat` | |
| `np.transpose(a, axes)` | `t.permute(*axes)` | `t.transpose(d0, d1)` swaps exactly two dims |

### 2.2 — dtype: why float32?

MATLAB defaults to `double` (64-bit). PyTorch defaults to **`float32`** (32-bit) — and deep learning wants it that way:

* Half the memory → bigger batches on the same GPU.
* GPUs execute float32 (and float16/bfloat16) dramatically faster than float64.
* Training is noise-tolerant; the extra 32 bits of precision buy nothing.

The visual below shows the memory cost per dtype for one batch of this project's data:

In [ ]:
import matplotlib.pyplot as plt

# One batch: (B=100, W=59, T=100, A=6, C=58)
elems = 100 * 59 * 100 * 6 * 58
dtypes = ["float64\n(MATLAB default)", "float32\n(PyTorch default)", "float16 / bfloat16\n(mixed precision)"]
bytes_per = [8, 4, 2]
gb = [elems * b / 1024**3 for b in bytes_per]

fig, ax = plt.subplots(figsize=(8, 3.6))
bars = ax.barh(dtypes[::-1], gb[::-1], color=["#e6f0d0", "#cce4ff", "#ffe4cc"])
for bar, val in zip(bars, gb[::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f} GB", va="center", fontsize=11, fontweight="bold")
ax.set_xlabel("memory for ONE batch of (100, 59, 100, 6, 58)")
ax.set_title("Same data, three dtypes — why float32 is the training default", fontsize=12)
ax.set_xlim(0, max(gb) * 1.25)
plt.tight_layout(); plt.show()
print(f"{elems:,} elements per batch — dtype choice is a {gb[0]:.1f} GB vs {gb[1]:.1f} GB decision.")

In [ ]:
# dtype conversions
t64 = torch.zeros(3, dtype=torch.float64)
t32 = t64.float()               # to float32
t16 = t64.half()                # to float16
i64 = torch.tensor([1, 2, 3])   # integer literals → int64 by default
print(t64.dtype, t32.dtype, t16.dtype, i64.dtype)

**The classic dtype bug when porting from MATLAB:** `.mat` files hold float64, so `torch.from_numpy(mat_data)` gives you a float64 tensor. Feed that to a float32 model and you get `RuntimeError: expected scalar type Float but found Double`. The fix is one call — `.float()` — and this codebase's datasets do it for you:

```python
# src/neural_data_decoding/data/mat_dataset.py — the last line of __getitem__'s feature path
features = torch.from_numpy(np.ascontiguousarray(features_np)).float()
```

`torch.from_numpy` shares memory with the NumPy array (zero-copy); the `.float()` then makes the (copying) dtype conversion. Order matters — sharing first, converting second.

### 2.3 — NumPy ↔ tensor conversion and memory sharing

In [ ]:
arr = np.arange(5, dtype=np.float32)
t = torch.from_numpy(arr)         # SHARES memory with arr
t[0] = 99.0
print("arr after modifying tensor:", arr)   # arr changed too!

In [ ]:
# .numpy() also shares (CPU tensors only)
back = t.numpy()
back[1] = -1
print("tensor after modifying array:", t)

# To break the link, copy explicitly:
independent = torch.tensor(arr)        # torch.tensor() always copies
independent[2] = 12345
print("arr unaffected:", arr)

**Summary of the sharing rules** (same view-vs-copy theme as [02.1 §2.5](02.1_numpy_vs_matlab_arrays.ipynb)):

| Conversion | Shares memory? |
|---|---|
| `torch.from_numpy(arr)` | ✅ yes |
| `cpu_tensor.numpy()` | ✅ yes |
| `torch.tensor(arr)` | ❌ always copies |
| `gpu_tensor.numpy()` | ✖ error — must `.cpu()` first |

### 2.4 — Devices: CPU, CUDA, MPS

A tensor lives on exactly one device. Operations require ALL operands on the same device — PyTorch never silently transfers.

In [ ]:
# Discover what's available on this machine
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")      # Apple-silicon GPU
else:
    device = torch.device("cpu")
print(f"best available device: {device}")

In [ ]:
x = torch.randn(3, 3)                 # created on CPU by default
print("x lives on:", x.device)

x_dev = x.to(device)                   # move (no-op if already there)
print("x_dev lives on:", x_dev.device)

y = x_dev @ x_dev                      # compute happens on that device
print("result lives on:", y.device)

back_on_cpu = y.cpu()                  # bring back for numpy / printing / saving
print("gathered:", back_on_cpu.device)

**MATLAB ↔ PyTorch device operations:**

| MATLAB | PyTorch |
|---|---|
| `gpuArray(X)` | `x.to("cuda")` |
| `gather(X)` | `x.cpu()` |
| `canUseGPU` | `torch.cuda.is_available()` |
| `gpuDevice` | `torch.cuda.get_device_properties(0)` |
| (no Apple-GPU support) | `torch.backends.mps.is_available()` |

**The device dance in a training loop** is always the same three steps: model to device once (`model.to(device)`), each batch to device inside the loop (`x = x.to(device)`), results back to CPU only when you need numpy/printing. On ACCRE the device will be CUDA; on an M-series Mac it's MPS; the code is identical.

### 2.5 — In-place operations: the trailing-underscore convention

PyTorch names every in-place variant with a trailing `_`:

In [ ]:
t = torch.ones(3)
u = t.add(1)      # returns a NEW tensor; t unchanged
print("after add   :", t, u)

t.add_(1)          # modifies t IN PLACE
print("after add_  :", t)

t.zero_()          # fill with zeros in place
print("after zero_ :", t)

This is the mutable/immutable lesson from [00.3](../00_orientation/00.3_the_matlab_to_python_mental_model.ipynb) surfacing in the API: PyTorch makes mutation *visible in the name*. In-place ops save memory but can break autograd if the overwritten value was needed for the backward pass — when in doubt, use the out-of-place form.

`tensor.clamp_()`, `weight.normal_()`, `tensor.add_()` — you now know exactly what the trailing underscore means everywhere you see it. (One near-miss to be aware of: `optimizer.zero_grad()` has a *medial* underscore — it isn't part of this tensor-method naming convention, and since PyTorch 2.0 it clears gradients by setting each `.grad` to `None` rather than zeroing in place.)

## Section 3 — The `neural_data_decoding` implementation

The dataset's `__getitem__` return path exercises everything from this notebook — numpy-to-tensor conversion, contiguity, dtype:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/data/mat_dataset.py").read_text().splitlines()
start = next(i for i, line in enumerate(src) if "def __getitem__" in line)
for i in range(start, min(start + 42, len(src))):
    print(f"{i + 1:3} | {src[i]}")

Things to spot:

* `torch.from_numpy(np.ascontiguousarray(features_np)).float()` — share, make contiguous, then convert to float32.
* `torch.from_numpy(self._labels[idx]).long()` — targets converted to int64 (`long`), which is what `cross_entropy` requires for class indices.
* No `.to(device)` here — device placement is the training loop's job, not the dataset's. Datasets always produce CPU tensors; the `DataLoader`'s workers can then parallelize loading.

## Section 4 — Hands-on exercises

### Exercise 4.1 — the dtype bug, live

Create a float64 numpy array, convert with `torch.from_numpy`, and feed it to `torch.nn.Linear(4, 2)`. Observe the exact error, then fix it.

In [ ]:
import torch, numpy as np
lin = torch.nn.Linear(4, 2)
x64 = np.random.standard_normal((3, 4))    # float64!
try:
    lin(torch.from_numpy(x64))
except RuntimeError as e:
    print(f"RuntimeError: {e}")

In [ ]:
# Reveal — one .float() fixes it:
out = lin(torch.from_numpy(x64).float())
print(out.shape, out.dtype)

### Exercise 4.2 — round-trip through the best device

Move a tensor to the best available device, square it there, bring it back, and confirm it matches the CPU computation.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
x = torch.randn(4, 4)
if torch.cuda.is_available():
    dev = torch.device("cuda")
elif torch.backends.mps.is_available():
    dev = torch.device("mps")
else:
    dev = torch.device("cpu")
y_dev = (x.to(dev) ** 2).cpu()
y_cpu = x ** 2
print("max abs difference:", (y_dev - y_cpu).abs().max().item())

## Section 5 — Common errors

### `RuntimeError: expected scalar type Float but found Double`

Float64 data hit a float32 model. Call `.float()` on the input (or `model.double()` if you truly want float64 — you don't, for training).

### `RuntimeError: Expected all tensors to be on the same device, but found at least two devices`

One operand is on CPU and another on GPU. Find the tensor that skipped its `.to(device)` — usually a freshly-created constant or a target loaded inside the loop.

### `TypeError: can't convert cuda:0 device type tensor to numpy`

`.numpy()` only works on CPU tensors. Chain `.cpu().numpy()`.

### `RuntimeError: view size is not compatible with input tensor's size and stride`

`view()` needs contiguous memory and your tensor isn't (usually after `permute`/`transpose`). Use `.reshape()` (which copies when needed) or `.contiguous().view(...)`.

### `RuntimeError: a leaf Variable that requires grad is being used in an in-place operation`

An in-place op (`_` method) collided with autograd. Use the out-of-place variant. Full story in [02.5 autograd basics](02.5_autograd_basics.ipynb).

### Everything is on GPU but training is SLOW

Usually the DataLoader is the bottleneck (loading .mat files per item), not the GPU. Or you're calling `.cpu()`/`.item()` inside the inner loop, forcing a device sync every iteration. Profile before assuming.

## Section 6 — Further reading

- [PyTorch tensor tutorial](https://pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html) — the official intro.
- [torch.Tensor docs](https://pytorch.org/docs/stable/tensors.html) — dtype table + every method.
- [CUDA semantics](https://pytorch.org/docs/stable/notes/cuda.html) — asynchronous execution, streams, memory caching.
- [MPS backend notes](https://pytorch.org/docs/stable/notes/mps.html) — Apple-silicon specifics.

Next up: **[02.5 autograd basics](02.5_autograd_basics.ipynb)** — the `requires_grad` flag, `.backward()`, and what MATLAB's `dlfeval`/`dlgradient` do under the hood.